In [3]:
import os
import pandas as pd
import dateutil
import numpy as np
import shutil
import glob

In [21]:
def get_filtered(subdir):
    df = pd.read_csv('output/' + subdir + '/' + subdir + '.csv', parse_dates=['start_time', 'end_time'])
    
    plot_dir = f'output/{subdir}/plots'
    fnames = os.listdir(plot_dir)
    fnames = [f for f in fnames if f.endswith('.png')]
    stimes = []
    for fname in fnames:
        toks = fname.split('_')[-2:]
        stime = dateutil.parser.parse(toks[0])
        stimes.append(stime)
        
    mask = df.start_time.apply(lambda s: s in stimes)
    df_filtered = df[mask]

    ordered_fnames = []
    for _, row in df_filtered.iterrows():
        fname = [f for f in fnames if row.start_time.isoformat() in f][0]
        ordered_fnames.append(fname)

    df_filtered['file_name'] = ordered_fnames
    df_filtered.reset_index(inplace=True)
    return df_filtered

In [24]:


def process(name):
    df = get_filtered(name)
    df['B'] = np.sqrt(df['Bx']**2 + df['By']**2 + df['Bz']**2)
    df['cone_angle'] =  np.rad2deg(np.arccos(df['Bx'] / df['B']))
    df['clock_angle'] =  np.rad2deg(np.arctan2(df['By'], df['Bz'])) % 360
    
    labels = []

    for _, row in df.iterrows():    
        cone_angle = row.cone_angle
        clock_angle = row.clock_angle 
        Bx = row.Bx
        B = row.B
        
        if np.abs(Bx) / B > 0.8:
            label = 'BigBx'
        elif clock_angle > 305 or clock_angle < 55:
            label = 'NorthwardIMF'
        elif (clock_angle > 55 and clock_angle < 155) or (clock_angle > 205 and clock_angle < 305):
            label = 'ByDominant'
        elif clock_angle > 155 and clock_angle < 205:
            label = 'SouthwardIMF'
        else:
            raise RuntimeError((cone_angle, clock_angle))
    
        labels.append(label)
        
    df['label'] = labels
    df = df.sort_values('start_time')

    def get_aci_file(row):
        tstamp = row.start_time.strftime("%Y%m%d")
        return glob.glob(f'data/{name}/aci/*{tstamp}*.cdf')[0]
    
    def get_ead_file(row):
        tstamp = row.start_time.strftime("%Y%m%d")
        return glob.glob(f'data/{name}/ead/*{tstamp}*.cdf')[0]

    
    df['aci_file'] = df.apply(get_aci_file, axis=1)
    df['ead_file'] = df.apply(get_ead_file, axis=1)

    df.to_csv(f'output/{name}_filtered.csv', index=0)

In [22]:
process('Sept30_Storm')

In [26]:
process('Nov11_Storm')

In [27]:
process('Jan19_Storm')